### 使用模型：Gemini 2.0 Pro

幫我撰寫一個 Python 原始碼，可以做到下列步驟：

*   抓取 https://udn.com/rank/pv/2 和 https://news.ltn.com.tw/list/breakingnews/popular
*   用 requests 套件，抓取這兩間報社的「熱門新聞」原始文字，並將之印出

撰寫此一程式時，請遵守下列限制：

*   將該程式碼撰寫成適合在 Google Colab 環境下執行的樣子。

In [1]:
import requests

def fetch_hot_news_html(url, newspaper_name):
  """
  抓取指定網址的熱門新聞原始碼，並印出。

  Args:
    url: 熱門新聞網址。
    newspaper_name: 報社名稱，用於輸出時的辨識。
  """
  try:
    response = requests.get(url)
    response.raise_for_status()  # 檢查請求是否成功 (狀態碼 200)

    print(f"\n--- {newspaper_name} 熱門新聞原始碼 ---")
    print(response.text)

  except requests.exceptions.RequestException as e:
    print(f"抓取 {newspaper_name} 熱門新聞時發生錯誤：{e}")

if __name__ == "__main__":
  udn_url = "https://udn.com/rank/pv/2"
  ltn_url = "https://news.ltn.com.tw/list/breakingnews/popular"

  fetch_hot_news_html(udn_url, "聯合新聞網 (UDN)")
  fetch_hot_news_html(ltn_url, "自由時報 (LTN)")


--- 聯合新聞網 (UDN) 熱門新聞原始碼 ---
<!-- nav:0.0000 --><!-- top:0.0165 --><!-- bottom:0.0010 --><!-- meta:0.0002 --><!-- ad:0.0177 --><!-- content:0.1751 --><!-- total:0.2526 --><!-- misc rank3pv 0.2526 --><!--hottest_news 0.0015 --><!--content 0.0114 --><!--ad_1 0.0263 --><!--ad 0.0518 --><!--view_type 0.0000 --><!--meta 0.0001 --><!--family_bar 0.0006 --><!--family_bar sub  --><!--breaking_news 0.0036 --><!--top 0.0068 --><!--bottom 0.0019 --><!--nav 0.0000 --><!--code 0.0722 --><!DOCTYPE html>
<html lang="zh-TW" itemscope itemtype="https://schema.org/WebSite">
<head>
    <title>最多瀏覽 | 聯合新聞網</title>  <title>最多瀏覽 | 聯合新聞網</title> 
<link rel="canonical" itemprop="mainEntityOfPage" href="https://udn.com/rank/pv/2"/>
<meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1">
<meta http-equiv="Content-Type" content="text/html; charset=utf-8">
<meta http-equiv="content-language" content="zh-Hant-TW"> 
<meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=5.0, mini

### 使用模型：Gemini 2.0 Pro
幫我撰寫一個 Python 原始碼，可以做到下列步驟：

*   抓取 https://udn.com/rank/pv/2 和 https://news.ltn.com.tw/list/breakingnews/popular
*   抓出每則熱門新聞的「標題」、「超連結」、「發表時間」，並標註「資料來源」，並整理成 JSON 格式印出。

撰寫此一程式時，請遵守下列限制：

*   將該程式碼撰寫成適合在 Google Colab 環境下執行的樣子。

In [2]:
import requests
from bs4 import BeautifulSoup
import json
from datetime import datetime

def scrape_udn_hot_news():
    """
    抓取聯合新聞網 (udn.com) 熱門新聞。
    (UDN 的程式碼部分保持更新，直接針對 'story-list__text' 容器迭代, 並修改時間格式)
    """
    url = "https://udn.com/rank/pv/2"
    try:
        response = requests.get(url)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, "html.parser")

        news_list = []
        for news_item_content in soup.find_all('div', class_='story-list__text'):
            try:
                title_tag = news_item_content.find('h2').find('a')
                time_tag = news_item_content.find('time', class_='story-list__time')

                if title_tag and time_tag:
                    title = title_tag.text.strip()
                    link = "https://udn.com" + title_tag['href']
                    time_str = time_tag.text.strip()

                    try:
                        # 修改時間格式化為 "YYYY-MM-DD HH:MM"
                        publish_time = datetime.strptime(time_str, "%Y-%m-%d %H:%M").strftime("%Y-%m-%d %H:%M")
                    except ValueError:
                        publish_time = None

                    news_list.append({
                        "標題": title,
                        "超連結": link,
                        "發表時間": publish_time,
                        "資料來源": "聯合新聞網 (udn.com)"
                    })
                else:
                    print("!!! UDN 標題或時間標籤未找到 !!!")
            except Exception as e:
                continue

        return news_list

    except requests.exceptions.RequestException as e:
        print(f"抓取聯合新聞網失敗: {e}")
        return []

def scrape_ltn_hot_news():
    """
    抓取自由時報 (ltn.com.tw) 熱門新聞。
    (自由時報程式碼修改標題處理 - 正確提取標題)
    """
    url = "https://news.ltn.com.tw/list/breakingnews/popular"
    try:
        response = requests.get(url)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, "html.parser")

        news_list = []
        news_items = soup.select('.whitecon .list li')

        if not news_items:
            print("!!! 自由時報 沒有抓取到任何新聞項目，可能選擇器有誤或網頁結構變更 !!!")
            return []

        for item in news_items:
            title_tag = item.select_one('a.tit')
            time_tag = item.select_one('.time')

            if title_tag and time_tag:
                title_parts = title_tag.text.strip().split('\n')
                title = title_parts[2] if len(title_parts) > 2 else (title_parts[1] if len(title_parts) > 1 else title_parts[0]) # 正確取第三部分，備用方案保持不變

                link = title_tag['href']
                time_str = time_tag.text.strip()

                try:
                    publish_time = datetime.strptime(time_str, "%Y/%m/%d %H:%M").strftime("%Y-%m-%d %H:%M")
                except ValueError:
                    publish_time = None

                news_list.append({
                    "標題": title,
                    "超連結": link,
                    "發表時間": publish_time,
                    "資料來源": "自由時報 (ltn.com.tw)"
                })
            else:
                print("!!! 自由時報 標題或時間標籤未找到 !!!")

        return news_list

    except requests.exceptions.RequestException as e:
        print(f"抓取自由時報失敗: {e}")
        return []

if __name__ == "__main__":
    udn_news = scrape_udn_hot_news()
    ltn_news = scrape_ltn_hot_news()

    all_news = udn_news + ltn_news

    json_output = json.dumps(all_news, ensure_ascii=False, indent=2)
    print(json_output)

[
  {
    "標題": "影／女大學生被騙到柬埔寨打工獲救回家抱阿公哭 她說明被騙原委",
    "超連結": "https://udn.comhttps://udn.com/news/story/124490/8792656",
    "發表時間": "2025-06-08 11:32",
    "資料來源": "聯合新聞網 (udn.com)"
  },
  {
    "標題": "獨／成大旁嚴重車禍！闖紅燈猛撞30多車「釀1死2傷」 76歲婦嚇壞稱誤踩油門",
    "超連結": "https://udn.comhttps://udn.com/news/story/7320/8793049",
    "發表時間": "2025-06-08 16:19",
    "資料來源": "聯合新聞網 (udn.com)"
  },
  {
    "標題": "汽車停自家騎樓遭檢舉挨罰 他提告：不良民眾浮濫檢舉",
    "超連結": "https://udn.comhttps://udn.com/news/story/7321/8792725",
    "發表時間": "2025-06-08 12:08",
    "資料來源": "聯合新聞網 (udn.com)"
  },
  {
    "標題": "主播搭商務車廂被盯上怨「像逃票的中學生」 高鐵說話了",
    "超連結": "https://udn.comhttps://udn.com/news/story/7266/8792660",
    "發表時間": "2025-06-08 11:33",
    "資料來源": "聯合新聞網 (udn.com)"
  },
  {
    "標題": "神經內科醫生不做的4種「傷腦行為」 你中了幾種？",
    "超連結": "https://udn.comhttps://udn.com/news/story/7266/8793211",
    "發表時間": "2025-06-08 19:35",
    "資料來源": "聯合新聞網 (udn.com)"
  },
  {
    "標題": "罷吳敗張克晉仍失聯 趙少康曝可能原因：是人不會這樣做",
    "超連結": "https://udn.comhttps://udn.

### 環境設定

#### MongoDB 伺服器

In [3]:
# 下載免費開源的 MongoDB 社群版
# 可從 https://www.mongodb.com/download-center/community/releases 查詢你要下載的版本
# 需與 Google Colab 當前使用的作業系統一致（Ubuntu 22.04）
!wget -qO - https://fastdl.mongodb.org/linux/mongodb-linux-x86_64-ubuntu2204-7.0.7.tgz | tar -xzv
!mv mongodb-linux-x86_64-ubuntu2204-7.0.7/bin/* /usr/bin/

mongodb-linux-x86_64-ubuntu2204-7.0.7/LICENSE-Community.txt
mongodb-linux-x86_64-ubuntu2204-7.0.7/MPL-2
mongodb-linux-x86_64-ubuntu2204-7.0.7/README
mongodb-linux-x86_64-ubuntu2204-7.0.7/THIRD-PARTY-NOTICES
mongodb-linux-x86_64-ubuntu2204-7.0.7/bin/install_compass
mongodb-linux-x86_64-ubuntu2204-7.0.7/bin/mongod
mongodb-linux-x86_64-ubuntu2204-7.0.7/bin/mongos


In [4]:
# 下載並安裝與 MongoDB 版本相應的金鑰，保證 MongoDB 連線的安全
# 可以上 https://pgp.mongodb.com/ 查詢欲下載金鑰的完整檔名
!wget https://www.mongodb.org/static/pgp/server-7.0.asc
!sudo apt-key add server-7.0.asc

--2025-06-09 03:34:53--  https://www.mongodb.org/static/pgp/server-7.0.asc
Resolving www.mongodb.org (www.mongodb.org)... 52.21.89.200, 52.206.222.245, 54.175.147.155
Connecting to www.mongodb.org (www.mongodb.org)|52.21.89.200|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://pgp.mongodb.com/server-7.0.asc [following]
--2025-06-09 03:34:53--  https://pgp.mongodb.com/server-7.0.asc
Resolving pgp.mongodb.com (pgp.mongodb.com)... 3.168.132.98, 3.168.132.14, 3.168.132.63, ...
Connecting to pgp.mongodb.com (pgp.mongodb.com)|3.168.132.98|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1674 (1.6K) [binary/octet-stream]
Saving to: ‘server-7.0.asc’

server-7.0.asc      100%[===================>]   1.63K  --.-KB/s    in 0s      

2025-06-09 03:34:53 (385 MB/s) - ‘server-7.0.asc’ saved [1674/1674]

OK


In [5]:
# 建造 MongoDB 資料庫存放處（路徑、名稱可自訂）
!mkdir -p /content/data/db

# 啟動 MongoDB 伺服器（Mongo Daemon）於本地端（localhost）
!mongod --fork --logpath /var/log/mongodb.log --dbpath /content/data/db --bind_ip localhost

about to fork child process, waiting until server is ready for connections.
forked process: 947
child process started successfully, parent exiting


In [6]:
# 掛載私人 Google Drive 儲存 MongoDB 資料庫資料
# from google.colab import drive
# drive.mount('/content/gdrive')

# 啟動 MongoDB 伺服器（Mongo Daemon）於本地端（localhost）（使用掛載的外部硬碟會耗時約 1 分鐘，請耐心等待）
# !mongod --fork --logpath /var/log/mongodb.log --dbpath /content/gdrive/MyDrive/Datasets/MongoDB --bind_ip localhost

In [7]:
# 確認 MongoDB 正在執行中
!mongod --sysinfo

{
    "sysinfo": {
        "_SC_PAGE_SIZE": 4096,
        "_SC_PHYS_PAGES": 3322356,
        "_SC_AVPHYS_PAGES": 1842433
    }
}


#### PyMongo 函式庫

In [8]:
# 安裝 PyMongo 函式庫
!pip install pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.6/313.6 kB 17.8 MB/s eta 0:00:00


### 使用模型：Gemini 2.0 Pro
假設 MongoDB 與 PyMongo 已經安裝完畢，且前面程式碼已經產生名為json_output的JSON字串。請幫我寫一段 Python 原始碼，能執行下列要求：

*   將json_output寫入 MongoDB 伺服器內的新資料庫 'news' 裡面的 'hot_news' 資料表中。
*   將這些資料挑出 '標題' 欄位、'超連結' 欄位、'發表時間' 欄位、與 '資料來源' 欄位，分別對應到「標題」、「網址」、「日期」與「來源」這四個字串，然後以下列格式印出：

> "標題：", document["標題"] \
> "日期：", document["發表時間"] \
> "網址：", document["超連結"] \
> "來源：", document["資料來源"] \
> "---------------------------------"

撰寫此一程式時，請遵守下列限制：

*   將該程式碼撰寫成適合在 Google Colab 環境下執行的樣子。

In [9]:
import pymongo
import json

# MongoDB 連線設定
mongo_uri = "mongodb://localhost:27017/"  # 預設 MongoDB 連線 URI，請確認您的 MongoDB 伺服器位置
database_name = "news"
collection_name = "hot_news"

try:
    # 連線到 MongoDB 伺服器
    client = pymongo.MongoClient(mongo_uri)
    db = client[database_name]
    hot_news_collection = db[collection_name]

    # 將 JSON 字串解析為 Python 物件
    news_data = json.loads(json_output)

    # 插入資料到 MongoDB
    if isinstance(news_data, list):
        hot_news_collection.insert_many(news_data)
    else:
        hot_news_collection.insert_one(news_data)

    # 讀取資料並印出指定格式
    news_cursor = hot_news_collection.find() # 讀取所有資料

    for document in news_cursor:
        print("標題：", document.get("標題")) # 使用 .get() 避免 Key 不存在時出錯
        print("日期：", document.get("發表時間"))
        print("網址：", document.get("超連結"))
        print("來源：", document.get("資料來源"))
        print("---------------------------------")

except pymongo.errors.ConnectionFailure as e:
    print(f"無法連線到 MongoDB 伺服器: {e}")
except Exception as e:
    print(f"發生錯誤: {e}")
finally:
    if client:
        client.close()

標題： 影／女大學生被騙到柬埔寨打工獲救回家抱阿公哭 她說明被騙原委
日期： 2025-06-08 11:32
網址： https://udn.comhttps://udn.com/news/story/124490/8792656
來源： 聯合新聞網 (udn.com)
---------------------------------
標題： 獨／成大旁嚴重車禍！闖紅燈猛撞30多車「釀1死2傷」 76歲婦嚇壞稱誤踩油門
日期： 2025-06-08 16:19
網址： https://udn.comhttps://udn.com/news/story/7320/8793049
來源： 聯合新聞網 (udn.com)
---------------------------------
標題： 汽車停自家騎樓遭檢舉挨罰 他提告：不良民眾浮濫檢舉
日期： 2025-06-08 12:08
網址： https://udn.comhttps://udn.com/news/story/7321/8792725
來源： 聯合新聞網 (udn.com)
---------------------------------
標題： 主播搭商務車廂被盯上怨「像逃票的中學生」 高鐵說話了
日期： 2025-06-08 11:33
網址： https://udn.comhttps://udn.com/news/story/7266/8792660
來源： 聯合新聞網 (udn.com)
---------------------------------
標題： 神經內科醫生不做的4種「傷腦行為」 你中了幾種？
日期： 2025-06-08 19:35
網址： https://udn.comhttps://udn.com/news/story/7266/8793211
來源： 聯合新聞網 (udn.com)
---------------------------------
標題： 罷吳敗張克晉仍失聯 趙少康曝可能原因：是人不會這樣做
日期： 2025-06-08 11:52
網址： https://udn.comhttps://udn.com/news/story/124323/8792698
來源： 聯合新聞網 (udn.com)
---------------------------